# A5 image captioning — core

Adapted from University of Michigan EECS 498/598, Deep Learning for Computer Vision (Winter 2022), by Justin Johnson and course staff.

**Static validation only; not executed.** Enable the run switches explicitly; generated files go to `core_results/`.

In [ ]:
from pathlib import Path
import os
import sys
import torch
import matplotlib.pyplot as plt
candidates = []
for base in (Path.cwd(), *Path.cwd().parents,
             Path('/content/drive/MyDrive/umich-dlcv-wi2022')):
    candidates.extend((base, base / 'assignments' / 'a5'))
ASSIGNMENT_DIR = next((p.resolve() for p in candidates if (p / 'rnn_lstm_captioning_core.py').is_file()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError('Open from the project/assignment directory, or mount the canonical Colab Drive project first.')
loaded_helpers = sys.modules.get('eecs598')
if loaded_helpers is not None:
    loaded_path = Path(loaded_helpers.__file__).resolve().parent
    if loaded_path != (ASSIGNMENT_DIR / 'eecs598').resolve():
        raise RuntimeError('Restart the kernel before switching assignments: eecs598 is already loaded from another assignment.')
sys.path.insert(0, str(ASSIGNMENT_DIR))
os.chdir(ASSIGNMENT_DIR)
from eecs598.utils import reset_seed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ASSIGNMENT_DIR / 'core_results' / 'rnn_lstm_captioning'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_OVERFIT = False
ALLOW_DOWNLOAD = False
reset_seed(0)
plt.rcParams['image.cmap'] = 'gray'
print('Outputs:', OUTPUT_DIR)

## COCO data and caption models
The reconstructed COCO2014 subset has the course tensor shapes and vocabulary size but differs from the unavailable original subset.

In [ ]:
from rnn_lstm_captioning_core import CaptioningRNN
from a5_helper_core import load_coco_captions, train_captioner, decode_captions
from prepare_coco_core import prepare_coco
from eecs598.utils import attention_visualizer
from torchvision import transforms
from torchvision.utils import make_grid
DATA_PATH = ASSIGNMENT_DIR / 'datasets' / 'coco.pt'
BATCH_SIZE, OVERFIT_BATCH_SIZE, VIS_BATCH_SIZE = 256, 32, 4
VARIANTS = ('rnn', 'lstm', 'attn')
def make_captioner(kind, vocabulary):
    return CaptioningRNN(cell_type=kind, word_to_idx=vocabulary,
                         input_dim=400, hidden_dim=512, wordvec_dim=256,
                         ignore_index=vocabulary['<NULL>']).to(DEVICE)

In [ ]:
if RUN_TRAINING or RUN_EVALUATION or RUN_OVERFIT:
    if not DATA_PATH.is_file():
        if not ALLOW_DOWNLOAD:
            raise FileNotFoundError(f'Place COCO data at {DATA_PATH}, or enable ALLOW_DOWNLOAD to reconstruct it.')
        prepare_coco(DATA_PATH)
    data_dict = load_coco_captions(str(DATA_PATH))
    vocabulary = data_dict['vocab']['token_to_idx']
    num_train = len(data_dict['train_images'])

## Optional 50-image overfit
All three recurrent variants use 80 epochs; LSTM uses learning rate 0.01, and RNN/attention use 0.001.

In [ ]:
if RUN_OVERFIT:
    for kind in VARIANTS:
        reset_seed(0)
        indices = torch.linspace(0, num_train - 1, steps=50).long()
        overfit_model = make_captioner(kind, vocabulary)
        train_captioner(overfit_model, data_dict['train_images'][indices],
                        data_dict['train_captions'][indices], num_epochs=80,
                        batch_size=OVERFIT_BATCH_SIZE,
                        learning_rate=1e-2 if kind == 'lstm' else 1e-3, device=DEVICE)
        del overfit_model

## Full RNN, LSTM, and attention-LSTM training
Each variant uses the ordinary notebook’s seed-0 resampling, 60 epochs, and learning rate 0.001. Core checkpoints and loss histories are separate files.

In [ ]:
if RUN_TRAINING:
    loss_histories = {}
    for kind in VARIANTS:
        reset_seed(0)
        indices = torch.randint(num_train, size=(num_train,))
        captioner = make_captioner(kind, vocabulary)
        captioner, losses = train_captioner(captioner, data_dict['train_images'][indices],
                                          data_dict['train_captions'][indices], num_epochs=60,
                                          batch_size=BATCH_SIZE, learning_rate=1e-3, device=DEVICE)
        torch.save(captioner.state_dict(), OUTPUT_DIR / f'{kind}_captioner.pt')
        loss_histories[f'{kind}_losses'] = losses
        del captioner
    torch.save(loss_histories, OUTPUT_DIR / 'rnn_lstm_attention_submission.pt')

## Captions and attention maps
Evaluate core checkpoints on fixed random training and validation samples. The ordinary caption submission contains losses, not model weights, so it cannot replace these checkpoints.

In [ ]:
if RUN_EVALUATION:
    reset_seed(0)
    for kind in VARIANTS:
        checkpoint = OUTPUT_DIR / f'{kind}_captioner.pt'
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Run full training first: {checkpoint}')
        captioner = make_captioner(kind, vocabulary)
        captioner.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True))
        captioner.eval()
        for split in ('train', 'val'):
            indices = torch.randint(len(data_dict[split + '_images']), (VIS_BATCH_SIZE,))
            images = data_dict[split + '_images'][indices]
            expected = decode_captions(data_dict[split + '_captions'][indices], data_dict['vocab']['idx_to_token'])
            with torch.no_grad():
                prediction = captioner.sample(images.to(DEVICE))
            tokens, attention = prediction if kind == 'attn' else (prediction, None)
            generated = decode_captions(tokens.cpu(), data_dict['vocab']['idx_to_token'])
            for i in range(VIS_BATCH_SIZE):
                fig, ax = plt.subplots(figsize=(7, 4))
                ax.imshow(images[i].permute(1, 2, 0))
                ax.set_title(f'{split} / {kind}: {generated[i]}\nGT: {expected[i]}')
                ax.axis('off')
                fig.savefig(OUTPUT_DIR / f'{kind}_{split}_{i}.jpg', bbox_inches='tight')
                plt.show()
                if attention is not None:
                    panels = [transforms.ToTensor()(attention_visualizer(images[i], attention[i, j].cpu(), token))
                              for j, token in enumerate(generated[i].split())]
                    grid = make_grid(panels, nrow=8)
                    fig, ax = plt.subplots(figsize=(16, 5))
                    ax.imshow(torch.flip(grid, dims=(0,)).permute(1, 2, 0))
                    ax.axis('off')
                    fig.savefig(OUTPUT_DIR / f'attention_{split}_{i}.jpg', bbox_inches='tight')
                    plt.show()
        del captioner